# TP3 - Análisis de Sentimiento sobre Tweets (Sentiment140)
## Notebook 4 — Predicciones con el Modelo Elegido
### Diplomatura IA - UP
### Alumno: Gonzalez Marta Elizabeth
### Mes: Julio26
Este notebook toma el modelo ya **entrenado, validado y evaluado** en los notebooks anteriores (`data/modelo_final_elegido.pkl`) y lo usa para:

1. Generar predicciones detalladas sobre el conjunto de test (con el texto original, la etiqueta real y la predicha).
2. Mostrar ejemplos concretos de aciertos y errores.
3. Predecir sobre **texto nuevo, arbitrario** — la función que se usaría en un caso de uso real (por ejemplo, un sistema que clasifica tweets nuevos a medida que llegan).

In [1]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import hstack, csr_matrix

with open('../data/processed/modelo_final_elegido.pkl', 'rb') as f:
    elegido = pickle.load(f)
nombre_modelo, modelo_final = elegido['nombre'], elegido['modelo']

with open('../data/processed/tfidf_vectorizer_mega.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open('../data/processed/scaler_features.pkl', 'rb') as f:
    scaler = pickle.load(f)

test = pd.read_csv('../data/processed/split_test.csv')

FEATURE_COLS = [
    'n_chars', 'n_words', 'avg_word_len', 'n_exclamation', 'n_question',
    'n_uppercase_words', 'n_elongated', 'n_pos_emoticons', 'n_neg_emoticons',
    'n_mentions_count', 'n_hashtags_count', 'n_lex_pos', 'n_lex_neg', 'lex_pos_neg_diff',
]

print(f'Modelo cargado: {nombre_modelo}')
print(f'Test: {test.shape}')


Modelo cargado: Regresión Logística + FE
Test: (477819, 30)


## 1. Predicciones detalladas sobre el conjunto de test

Reconstruimos la matriz de features correspondiente al tipo de modelo elegido (algunos usan solo TF-IDF, otros TF-IDF + Feature Engineering).

In [2]:
Xtest_tfidf = tfidf.transform(test['clean_text'])

if nombre_modelo == 'Regresión Logística + FE':
    Xtest_feat = scaler.transform(test[FEATURE_COLS])
    Xtest_final = hstack([Xtest_tfidf, csr_matrix(Xtest_feat)])
else:
    Xtest_final = Xtest_tfidf

pred = modelo_final.predict(Xtest_final)

resultados = test[['text', 'target']].copy()
resultados['prediccion'] = pred
resultados['correcto'] = resultados['target'] == resultados['prediccion']

mapa_etiquetas = {0: 'Negativo', 2: 'Neutral', 4: 'Positivo'}
resultados['target_label'] = resultados['target'].map(mapa_etiquetas)
resultados['prediccion_label'] = resultados['prediccion'].map(mapa_etiquetas)

print(f'Accuracy sobre test: {resultados["correcto"].mean():.2%}')
resultados.head(10)


Accuracy sobre test: 80.69%


,text,target,prediccion,correcto,target_label,prediccion_label
0,My TV needs to stop tormenting me with the GI ...,0,0,True,Negativo,Negativo
1,@shortformblog Alright thanks. Much appreciated.,4,4,True,Positivo,Positivo
2,@Renee3 It means were still kids at heart. Don...,4,4,True,Positivo,Positivo
3,Has so many emotions about tonight,0,0,True,Negativo,Negativo
4,@Depond sadly no ive got to work,0,0,True,Negativo,Negativo
5,"@TraceCyrus That is sooo true, you have good a...",4,4,True,Positivo,Positivo
6,"@thisisryanross feel better, Ryan!",4,4,True,Positivo,Positivo
7,@CombustionGlass the air france thing is freak...,0,0,True,Negativo,Negativo
8,@macbuddydev Its pretty good.,4,4,True,Positivo,Positivo
9,www.plattenm sli.de ist online (via @infinif0ld),4,4,True,Positivo,Positivo


## 2. Ejemplos de aciertos

In [3]:
aciertos = resultados[resultados['correcto']].sample(min(5, resultados['correcto'].sum()), random_state=42)
for _, row in aciertos.iterrows():
    print(f"[{row['target_label']} -> {row['prediccion_label']}] {row['text'][:100]}")


[Positivo -> Positivo] hey pipz 
[Positivo -> Positivo] Can't help but fall asleep smiling  good night
[Negativo -> Negativo] @pyroezra but you should move here! I have school. 
[Negativo -> Negativo] Is at work 
[Positivo -> Positivo] loves @bendennis video chats &amp; can't wait for @julespari to wake up tomorrow &amp; see the BBM's


## 3. Ejemplos de errores

In [4]:
errores = resultados[~resultados['correcto']].sample(min(5, (~resultados['correcto']).sum()), random_state=42)
for _, row in errores.iterrows():
    print(f"[real={row['target_label']} | predicho={row['prediccion_label']}] {row['text'][:100]}")


[real=Negativo | predicho=Positivo] @FrazJ i thought you had stopped  when are you going to see katy perry?
[real=Negativo | predicho=Positivo] step one week of the AMAZING! concert of @Jonasbrothers in argentina! I want another concert again !
[real=Positivo | predicho=Negativo] How can there be adverts already xD. The camera keeps panning round to Rob though *dies* 
[real=Negativo | predicho=Positivo] @TamekaRaymond simple yet strong...it touches on just what i'm going through. but i kind of want my 
[real=Negativo | predicho=Positivo] @Angry_Betta i know!! I was so excited with the weather report today...then we didn't get anything 


## 4. Función de predicción para texto nuevo

Esta es la función que se usaría en producción: recibe un tweet nuevo (que el modelo nunca vio), lo limpia con la misma metodología del Notebook 1, y devuelve la predicción.

In [5]:
import re
import html
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor'}
lemmatizer = WordNetLemmatizer()

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#(\w+)')
NONALPHA_RE = re.compile(r"[^a-zA-Z\s']")
MULTISPACE_RE = re.compile(r'\s+')

def limpiar_texto(text):
    text = str(text)
    text = html.unescape(text)
    text = URL_RE.sub(' ', text)
    text = MENTION_RE.sub(' ', text)
    text = HASHTAG_RE.sub(r'\1', text)
    text = text.lower()
    text = NONALPHA_RE.sub(' ', text)
    text = MULTISPACE_RE.sub(' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

def predecir_sentimiento(texto_nuevo):
    texto_limpio = limpiar_texto(texto_nuevo)
    X_nuevo = tfidf.transform([texto_limpio])

    if nombre_modelo == 'Regresión Logística + FE':
        # Para un caso de uso real, estas features se calcularían igual que en el Notebook 1
        n_chars = len(texto_nuevo)
        n_words = len(texto_nuevo.split())
        feats = np.array([[n_chars, n_words, 0, texto_nuevo.count('!'), texto_nuevo.count('?'),
                            0, 0, 0, 0, 0, 0, 0, 0, 0]])
        feats_scaled = scaler.transform(feats)
        X_nuevo = hstack([X_nuevo, csr_matrix(feats_scaled)])

    pred = modelo_final.predict(X_nuevo)[0]
    return mapa_etiquetas[pred]

# Ejemplos de prueba con texto inventado
ejemplos_nuevos = [
    "I love this so much, best day ever!!!",
    "This is the worst experience I've ever had, I hate it",
    "The meeting is scheduled for 3pm tomorrow",
]

for texto in ejemplos_nuevos:
    print(f'"{texto}" -> {predecir_sentimiento(texto)}')


"I love this so much, best day ever!!!" -> Positivo
"This is the worst experience I've ever had, I hate it" -> Negativo
"The meeting is scheduled for 3pm tomorrow" -> Negativo


C:\Users\Usuario\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Usuario\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\Usuario\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


**Nota:** para el modelo "Regresión Logística + FE", varias features (mayúsculas, elongaciones, emoticones, lexicón) se dejaron en 0 en este ejemplo simplificado — pero en un uso real conviene reutilizar exactamente las mismas funciones del Notebook 1 (`contar_mayusculas`, `contar_lexicon`, etc.) para calcularlas correctamente sobre el texto nuevo.

## 5. Guardado de las predicciones

In [6]:
resultados.to_csv('../data/processed/predicciones_test.csv', index=False)
print('Guardado: ../data/processed/predicciones_test.csv ->', resultados.shape)


Guardado: ../data/processed/predicciones_test.csv -> (477819, 6)


## 6. Resumen — Notebook 4

1. Se generaron predicciones detalladas sobre el conjunto de test usando el modelo elegido en el Notebook 3, con ejemplos concretos de aciertos y errores.
2. Se armó una función de predicción reutilizable (`predecir_sentimiento`) para clasificar texto nuevo, aplicando la misma limpieza del Notebook 1.
3. Las predicciones completas se exportaron a `../data/processed/predicciones_test.csv`.

**Continúa en `05_topicos_embeddings_grafos.ipynb`** (análisis complementario: tópicos, embeddings, grafos de usuarios).